# IMDB Sentiment Analysis — MLP From Scratch
**Objective:** Train a Multi-Layer Perceptron (MLP) **from scratch** with gradient descent to classify IMDB reviews.

**Features:** Sentence-level **TextBlob polarity only** (1 feature). No word-level scores, no padded vectors.

**V2 upgrades:** Mini-batch GD, Early stopping, L2 regularization (optional), Error analysis.


In [ ]:
# 1) Imports & configuration
import os
import re
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from textblob import TextBlob
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

DATA_PATH = r"/mnt/data/IMDB Dataset.csv"
SAMPLE_SIZE = None   # set e.g. 10000 for quick tests
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:
# 2) Load dataset
df = pd.read_csv(DATA_PATH)
df.columns = [c.strip().lower() for c in df.columns]
df.head()


## 3) EDA

In [ ]:
df.info()

In [ ]:
df['sentiment'].value_counts()

In [ ]:
df.isna().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
# df = df.drop_duplicates().reset_index(drop=True)  # optional

## 4) Text preprocessing

In [ ]:
\
def preprocess_text(text: str) -> str:
    t = str(text).lower()
    t = re.sub(r"<.*?>", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

df["review_clean"] = df["review"].apply(preprocess_text)
df[["review", "review_clean"]].head()


## 5) Feature engineering (TextBlob polarity only)

In [ ]:
def textblob_polarity(text: str) -> float:
    return float(TextBlob(text).sentiment.polarity)

if SAMPLE_SIZE is not None and SAMPLE_SIZE < len(df):
    df_work = df.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)
else:
    df_work = df.copy()

df_work["polarity"] = df_work["review_clean"].apply(textblob_polarity)

X = df_work[["polarity"]].to_numpy(dtype=np.float64)
y = df_work["sentiment"].map({"negative": 0, "positive": 1}).to_numpy(dtype=np.int64)

X.shape, y.shape, df_work["polarity"].describe()


## 6) Split (70/15/15)

In [ ]:
X_train, X_temp, y_train, y_temp, idx_train, idx_temp = train_test_split(
    X, y, np.arange(len(X)),
    test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
X_val, X_test, y_val, y_test, idx_val, idx_test = train_test_split(
    X_temp, y_temp, idx_temp,
    test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

X_train.shape, X_val.shape, X_test.shape


## 7) MLP from scratch (mini-batch GD + early stopping)

In [ ]:
def sigmoid(z):
    z = np.clip(z, -50, 50)
    return 1.0 / (1.0 + np.exp(-z))

def relu(z):
    return np.maximum(0.0, z)

def relu_grad(z):
    return (z > 0.0).astype(np.float64)

def bce_loss(y_true, y_pred, eps=1e-12):
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

def init_params(input_dim, hidden_dim):
    W1 = np.random.randn(input_dim, hidden_dim) * np.sqrt(2.0 / input_dim)
    b1 = np.zeros((1, hidden_dim))
    W2 = np.random.randn(hidden_dim, 1) * np.sqrt(2.0 / hidden_dim)
    b2 = np.zeros((1, 1))
    return {"W1": W1, "b1": b1, "W2": W2, "b2": b2}

def forward(X, params):
    Z1 = X @ params["W1"] + params["b1"]
    A1 = relu(Z1)
    Z2 = A1 @ params["W2"] + params["b2"]
    Yhat = sigmoid(Z2)
    return Yhat, {"Z1": Z1, "A1": A1, "Yhat": Yhat}

def compute_grads(X, y, params, cache, l2=0.0):
    N = X.shape[0]
    yhat = cache["Yhat"]
    Z1, A1 = cache["Z1"], cache["A1"]

    dZ2 = (yhat - y) / N
    dW2 = A1.T @ dZ2
    db2 = np.sum(dZ2, axis=0, keepdims=True)

    dA1 = dZ2 @ params["W2"].T
    dZ1 = dA1 * relu_grad(Z1)
    dW1 = X.T @ dZ1
    db1 = np.sum(dZ1, axis=0, keepdims=True)

    if l2 > 0:
        dW2 += (l2 / N) * params["W2"]
        dW1 += (l2 / N) * params["W1"]

    return {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2}

def update(params, grads, lr):
    params["W1"] -= lr * grads["dW1"]
    params["b1"] -= lr * grads["db1"]
    params["W2"] -= lr * grads["dW2"]
    params["b2"] -= lr * grads["db2"]
    return params

def minibatches(X, y, batch_size, rng):
    n = X.shape[0]
    idx = rng.permutation(n)
    for s in range(0, n, batch_size):
        b = idx[s:s+batch_size]
        yield X[b], y[b]

def train_mlp(X_train, y_train, X_val, y_val,
              hidden_dim=32, lr0=0.05, epochs=300, batch_size=512,
              l2=1e-3, patience=25, lr_decay=0.98):
    rng = np.random.default_rng(RANDOM_STATE)
    y_train = y_train.reshape(-1, 1).astype(np.float64)
    y_val = y_val.reshape(-1, 1).astype(np.float64)

    params = init_params(X_train.shape[1], hidden_dim)
    train_losses, val_losses = [], []
    best_params, best_val, best_epoch = None, float("inf"), 0
    no_improve = 0
    lr = lr0

    for epoch in range(1, epochs + 1):
        for Xb, yb in minibatches(X_train, y_train, batch_size, rng):
            yhat_b, cache_b = forward(Xb, params)
            grads = compute_grads(Xb, yb, params, cache_b, l2=l2)
            params = update(params, grads, lr=lr)

        yhat_tr, _ = forward(X_train, params)
        tr_loss = bce_loss(y_train, yhat_tr) + (l2/(2*len(X_train))) * (np.sum(params["W1"]**2) + np.sum(params["W2"]**2))

        yhat_va, _ = forward(X_val, params)
        va_loss = bce_loss(y_val, yhat_va) + (l2/(2*len(X_train))) * (np.sum(params["W1"]**2) + np.sum(params["W2"]**2))

        train_losses.append(tr_loss)
        val_losses.append(va_loss)

        if va_loss < best_val - 1e-6:
            best_val = va_loss
            best_epoch = epoch
            best_params = {k: v.copy() for k, v in params.items()}
            no_improve = 0
        else:
            no_improve += 1

        lr *= lr_decay

        if epoch % 25 == 0 or epoch == 1:
            print(f"Epoch {epoch:03d} | lr={lr:.5f} | train_loss={tr_loss:.4f} | val_loss={va_loss:.4f} | no_improve={no_improve}")

        if no_improve >= patience:
            print(f"Early stopping at epoch {epoch} (best epoch {best_epoch}, best val={best_val:.4f})")
            break

    return best_params, train_losses, val_losses, best_val, best_epoch

HIDDEN_DIM = 32
LR0 = 0.05
EPOCHS = 300
BATCH_SIZE = 512
L2 = 1e-3
PATIENCE = 25
LR_DECAY = 0.98

best_params, train_losses, val_losses, best_val_loss, best_epoch = train_mlp(
    X_train, y_train, X_val, y_val,
    hidden_dim=HIDDEN_DIM, lr0=LR0, epochs=EPOCHS, batch_size=BATCH_SIZE,
    l2=L2, patience=PATIENCE, lr_decay=LR_DECAY
)

best_val_loss, best_epoch


## 8) Test evaluation (threshold 0.5)

In [ ]:
test_probs, _ = forward(X_test, best_params)
y_pred = (test_probs.flatten() >= 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
print("Test accuracy:", round(acc, 4))
print(classification_report(y_test, y_pred, target_names=["negative","positive"], digits=4))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["negative","positive"])
disp.plot()
plt.show()


## 9) Loss curves

In [ ]:
plt.plot(train_losses, label="train")
plt.plot(val_losses, label="val")
plt.xlabel("Epoch")
plt.ylabel("BCE (+ L2)")
plt.title("Loss Curves (V2)")
plt.legend()
plt.show()


## 10) Error analysis

In [ ]:
df_test = df_work.iloc[idx_test].copy()
df_test["prob_positive"] = test_probs.flatten()
df_test["y_true"] = y_test
df_test["y_pred"] = y_pred
df_test["correct"] = (df_test["y_true"] == df_test["y_pred"])

mistakes = df_test[~df_test["correct"]].head(10)[["polarity","prob_positive","sentiment","y_pred","review_clean"]]
mistakes


## 11) Save artifacts

In [ ]:
import os, pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

OUT_DIR = os.path.abspath("..")
os.makedirs(os.path.join(OUT_DIR, "model"), exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "results"), exist_ok=True)

model_path = os.path.join(OUT_DIR, "model", "best_model.pkl")
with open(model_path, "wb") as f:
    pickle.dump({
        "params": best_params,
        "hidden_dim": HIDDEN_DIM,
        "lr0": LR0,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "l2": L2,
        "patience": PATIENCE,
        "lr_decay": LR_DECAY,
        "best_epoch": best_epoch,
        "feature": "TextBlob polarity (sentence-level, 1 feature)"
    }, f)

metrics_path = os.path.join(OUT_DIR, "results", "metrics.txt")
with open(metrics_path, "w", encoding="utf-8") as f:
    f.write("IMDB Sentiment Analysis — MLP From Scratch (V2)\n")
    f.write(f"Test accuracy: {acc:.4f}\n")
    f.write(f"Best val loss: {best_val_loss:.6f} at epoch {best_epoch}\n\n")
    f.write("Hyperparameters:\n")
    f.write(f"  hidden_dim={HIDDEN_DIM}, lr0={LR0}, epochs={EPOCHS}, batch_size={BATCH_SIZE}, l2={L2}, patience={PATIENCE}, lr_decay={LR_DECAY}\n\n")
    f.write("Classification report:\n")
    f.write(classification_report(y_test, y_pred, target_names=["negative","positive"], digits=4))
    f.write("\nConfusion matrix (rows=true, cols=pred):\n")
    f.write(np.array2string(cm))

cm_path = os.path.join(OUT_DIR, "results", "confusion_matrix.png")
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["negative","positive"])
disp.plot()
plt.savefig(cm_path, dpi=200, bbox_inches="tight")
plt.close()

loss_path = os.path.join(OUT_DIR, "results", "loss_curves.png")
plt.plot(train_losses, label="train")
plt.plot(val_losses, label="val")
plt.xlabel("Epoch")
plt.ylabel("BCE (+ L2)")
plt.title("Loss Curves (V2)")
plt.legend()
plt.savefig(loss_path, dpi=200, bbox_inches="tight")
plt.close()

discussion_path = os.path.join(OUT_DIR, "results", "results_discussion.txt")
with open(discussion_path, "w", encoding="utf-8") as f:
    f.write(
        "Results Discussion (V2)\n"
        "- Feature: TextBlob polarity only (sentence-level).\n"
        "- Training: mini-batch gradient descent, early stopping on validation loss.\n"
        "- Regularization: L2 penalty to reduce overfitting.\n"
        "- Errors: sarcasm / mixed sentiment.\n"
    )

mistakes_path = os.path.join(OUT_DIR, "results", "mistakes_sample.txt")
with open(mistakes_path, "w", encoding="utf-8") as f:
    f.write(mistakes.to_string(index=False))

submission_path = os.path.join(OUT_DIR, "submission.csv")
submission_df = pd.DataFrame({
    "id": idx_test,
    "sentiment_pred": np.where(y_pred == 1, "positive", "negative"),
    "prob_positive": test_probs.flatten()
}).sort_values("id")
submission_df.to_csv(submission_path, index=False)

print("Saved:", model_path, metrics_path, cm_path, loss_path, discussion_path, mistakes_path, submission_path)
